# Défi 2 — Administration territoriale du Togo
## Notebook Data Science & BI → Dashboard Streamlit

**Question centrale.** *Quels territoires ont aujourd'hui les besoins les plus importants en population, accès aux services de base et équipements publics, et comment les prioriser objectivement ?*

**Cahier des charges (5 objectifs).**
1. Profil démographique des régions, préfectures, communes **et cantons** (population 2022, poids relatif, tailles).
2. Évolution 2010–2022, **en tenant compte des changements de découpage**.
3. Inégalités d'accès : taux d'électricité par préfecture, agences CEET et TdE rapportées à la population.
4. Évolution de l'enregistrement des naissances et de la production de CNI / passeports.
5. Recommandations ciblées pour les territoires les moins bien servis.

**Fil narratif.** Population → Évolution → Accès aux services → Inégalités → Croisement des déficits → Priorisation → Recommandations.

**Avancement.**

| Chap. | Contenu | Statut |
|---|---|---|
| 00–01 | Introduction, exploration (contrôles du cahier d'analyse) | ✔ |
| 02–03 | Nettoyage, référentiel territorial, contrôles qualité | ✔ |
| 04–05 | Démographie 2022, cantons, évolution 2010–2022 (crosswalk) | ✔ |
| 06 | Services de base : électricité (taux ≠ personnes), agences CEET et TdE | ✔ |
| 07–08 | Indice de priorité territoriale, scénarios, corrélations, robustesse | ✔ |
| 09 | Recommandations chiffrées et messages clés | ✔ |
| Annexe A | Charte du template et contrastes | ✔ |

**Application.** Le tableau de bord (`app.py`, dossier `views/`) est construit sur le *template-modele* : il lit uniquement `outputs/data/`, produit par `tools/build_tables.py` à partir de ce même code (`utils/`).

**Étiquettes des données dérivées :** `OBSERVATION` · `CALCUL` · `ESTIMATION` · `PROXY` · `HYPOTHÈSE`. Les anomalies sont **conservées** (valeur brute) et consignées dans `anomaly_log`, jamais corrigées en silence.


In [1]:
import sys, platform, warnings
sys.path.insert(0, "..")                      # racine du projet (chemins relatifs uniquement)
import numpy as np, pandas as pd, plotly, scipy, sklearn, pyarrow
from IPython.display import display

from utils.io import load_data, OUT, fmt_int
from utils.quality import audit_data, clean_data, run_quality_tests, export_outputs, TIMINGS
from utils.demography import compute_demography, SEUIL_PETIT_CANTON, gini
from utils.evolution import build_crosswalk_2010_2022, detecter_transferts, tcam, SEUIL_FLAG_HIGH
from utils.export import export_demo_evolution
from utils.viz import plot_classement, plot_lorenz, plot_boxplot_tailles, plot_tcam, plot_contributions, plot_regions_evolution
from utils.theme import table_contrastes, PRIORITE, texte_lisible, contraste, fmt_int as fi, fmt_pct as fp, fmt_dec as fd
from IPython.display import Markdown
import time, tempfile, pathlib
from utils.services import construire_fact_prefecture, construire_fact_unite, TAUX_NATIONAL
from utils.priority import (COMPOSANTES, SCENARIOS, SCENARIO_DEFAUT, CODES, compute_ipt, robustesse, correlation_composantes, SEUIL_ROBUSTE)
from utils.story import construire_recommandations, messages_cles, phrase_sous_moyenne
from utils.formats import fmt_int as FI, fmt_dec as FD, fmt_pct as FP, fmt_pp, liste_fr
TMP = pathlib.Path(tempfile.mkdtemp())   # les exports du notebook vont dans un dossier temporaire : outputs/data n'est écrit que par tools/build_tables.py

RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120); pd.set_option("display.max_colwidth", 90)

print("Versions :", f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | plotly {plotly.__version__}"
      f" | scipy {scipy.__version__} | scikit-learn {sklearn.__version__} | pyarrow {pyarrow.__version__}")
print("random_state =", RANDOM_STATE)

Versions : Python 3.12.3 | pandas 3.0.2 | numpy 2.4.4 | plotly 7.1.0 | scipy 1.17.1 | scikit-learn 1.8.0 | pyarrow 26.0.0
random_state = 42


---
# Chapitre 01 — Exploration et contrôle du cahier d'analyse

**Question.** Que contiennent réellement les fichiers, et les affirmations du cahier d'analyse (§2 du prompt) sont-elles exactes ?
**Données.** 5 fichiers `observationdata-*`, 2 extraits d'agences (CEET, TdE) et 2 fichiers de métadonnées (204 champs). Les versions 2026 des agences sont **identiques hors identifiant** `FID` aux CSV 2024 : ce sont les CSV 2024 qui sont utilisés.
**Méthode.** Chargement en texte (aucune conversion implicite), inventaire, puis 67 contrôles automatisés (`outputs/etape0/etape0_exploration.py`).

In [2]:
raw = load_data()
inventaire = audit_data(raw)
display(inventaire)
print("NB : `pop2010` affiche 25 % de vides car la colonne `Unit` est entièrement vide (aucun impact).")

,fichier,lignes,colonnes,annees,pct_vides,lignes_dupliquees
0,pop2022,759,6,2022–2022,0.00,0
1,pop2010,555,4,2010–2010,25.00,0
2,elec,282,6,2018–2021,0.00,0
3,etat_civil,184,5,2014–2019,0.00,0
4,identite,33,5,2014–2022,0.00,0
5,ceet,34,14,,0.00,0
6,tde,59,14,,0.00,0
7,meta_ceet,204,5,,0.59,0
8,meta_tde,204,5,,0.59,0


NB : `pop2010` affiche 25 % de vides car la colonne `Unit` est entièrement vide (aucun impact).


In [3]:
v = pd.read_csv(pathlib.Path("..") / "outputs" / "etape0" / "verification_affirmations.csv")
print("Résultat des 67 contrôles de l'étape 0 :", v.statut.value_counts().to_dict())
display(v[v.statut.isin(["ÉCART", "NUANCE"])][["section", "affirmation", "obtenu"]])

Résultat des 67 contrôles de l'étape 0 : {'CONFIRMÉ': 58, 'ÉCART': 6, 'INFO': 2, 'NUANCE': 1}


,section,affirmation,obtenu
26,§2.2,[NOUVEAU] reconstitution Agoé-Nyivé par préfixes `AGOENYIVE =` + `VAKPOSSITO =` = flag...,"2010=285 404 → 2022=882 695 (×3.1, TCAM 9.9 %/an) vs Grand Lomé 2.8 %/an"
28,§2.3,Abonnés BT : 2018–2020 seulement,"2018–2020 pour 40 unités ; 2021 pour seulement 3 d'entre elles (AKEBOU, TANDJOARE, KPE..."
37,§2.3,[NOUVEAU] cohérence taux régionaux 2021 vs moyenne pondérée des unités (pop. 2022),SAVANES: 23.3 vs 22.39 (+0.9 pp); KARA: 36.5 vs 36.65 (-0.2 pp); CENTRALE: 37.2 vs 35....
42,§2.4,[NOUVEAU] variantes de `Nsp` dans etab_creation_date,"[""'Nsp'"", ""'Nsp '""] → [32, 11]"
49,§2.4,[NOUVEAU] mêmes comptes avec 'agence OUVERTE' (public + Utilisé) = définition de c6 §7,18 / 9 / 7
52,§2.5,[NOUVEAU] un libellé brut porte DEUX séries (« enfants scolarisés ne disposant pas d'a...,12 séries : A lignes 150–167 (313 346–411 310) ; B lignes 168–185 (0–3 219)
66,§2.6,"Pic 2019 (281 285), « creux 2020 » (236 212), 2022 : 219 331",2019=281285 / 2020=236212 / 2022=219331


**Lecture.** 58 affirmations confirmées ; 6 écarts et 1 nuance, aucun bloquant. Les plus structurants : (1) un libellé de l'état civil porte **deux séries** ; (2) « sans agence » a **deux définitions** (15/6/4 si on compte toute ligne, 18/9/7 si on exige une agence *ouverte*) ; (3) le périmètre d'**Agoè-Nyivé 2010** n'est pas reconstituable (×3,1) ; (4) les taux régionaux d'électricité **ne se recalculent pas** à partir des unités.

**Décisions validées (A–D).** (A) « sans agence » = pas d'agence *ouverte* (publique et `Utilisé`), variante `Nsp` en sensibilité ; (B) croissance `c3` neutre pour Golfe et Agoè-Nyivé pris séparément ; (C) série B de l'état civil hors analyse ; (D) `N/a` = non ouverte, `Nsp` = inconnue.

---
# Chapitre 02 — Nettoyage et référentiel territorial

**Question.** Peut-on fabriquer un référentiel territorial *fiable* (région → préfecture → commune → canton) à partir de listes plates sans colonne de hiérarchie ?
**Données.** `observationdata-kwwolwb.csv` (2022, 759 lignes), `observationdata-sapxctg.csv` (2010, 555 lignes), + noms accentués des fichiers d'agences.
**Méthode.** Les niveaux sont **reconstruits par l'ordre des lignes puis validés par somme** (parent = somme des enfants). Toute jointure passe par une **clé de chemin** `region|prefecture|commune|canton` (jamais le nom seul). Les corrections de libellés sont **journalisées**.

In [4]:
ctx = clean_data(raw)
print(f"clean_data : {TIMINGS['clean_data']:.2f} s")
dim, d10 = ctx["dim_territoire"], ctx["dim_2010"]

niveaux = pd.concat({
    "2022": dim.groupby("niveau").size(),
    "2010": d10.groupby("niveau").size()}, axis=1).fillna(0).astype(int)
display(niveaux)
print("Cantons hors DAGL :", int((dim.niveau_feuille == "canton").sum()), "| quartiers du Grand Lomé (DAGL) :", int((dim.niveau_feuille == "quartier").sum()))

clean_data : 0.46 s


,2022,2010
niveau,,
canton,597,0
commune,116,0
national,1,1
prefecture,39,36
region,6,6
arrondissement,0,5
feuille,0,507


Cantons hors DAGL : 375 | quartiers du Grand Lomé (DAGL) : 222


**Lecture.** 6 régions, 39 préfectures, 116 lignes-communes (117 communes : `DANYI 1+DANYI 2` est une ligne unique pour deux communes), 375 cantons hors Grand Lomé et 222 quartiers dans le DAGL. En 2010 : 6 blocs (Lomé séparé de la Maritime), 36 préfectures et 5 arrondissements de Lomé.
**So what.** Dans le Grand Lomé les feuilles sont des **quartiers** : on les traite à part (`niveau_feuille`) pour ne pas biaiser les statistiques sur les cantons.

In [5]:
corr = pd.DataFrame(ctx["corrections_2022"] + ctx["corrections_2010"])
print(f"{len(corr)} corrections / signalements de libellés (détail complet dans anomaly_log) :")
display(corr[["ligne", "valeur", "nature", "traitement"]])

18 corrections / signalements de libellés (détail complet dans anomaly_log) :


,ligne,valeur,nature,traitement
0,198,BINAH2,Espace manquant,Renommée « BINAH 2 »
1,407,AMOU 2,Libellé dupliqué dans la préfecture (2e occurrence = commune n°3),Renommée « AMOU 3 » (confirmé par la somme des préfectures)
2,432,EST-MONO 1,Libellé dupliqué dans la préfecture (2e occurrence = commune n°3),Renommée « EST-MONO 3 » (confirmé par la somme des préfectures)
3,512,TOTOAL AVE,Faute de frappe (préfixe « TOTOAL »),Renommée « AVE » (préfecture d'Avé)
4,366,DANYI 1+DANYI 2,Une seule ligne pour 2 communes (DANYI 1 et DANYI 2),"Conservée telle quelle, drapeau `commune_fusionnee` ; non séparable sans inventer"
5,547,KELEGOUGAN1,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
6,548,ATIEGOU1,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
7,551,KELEGOUGAN 2,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
8,558,ATIEGOU 2,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
9,567,KELEGOUGAN 3,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)


In [6]:
pref = dim[dim.niveau == "prefecture"][["nom", "region_nom", "region_5_nom", "population_2022", "unite_electrique", "taux_herite_du_groupe"]]
print("Les 39 préfectures du référentiel (région 2022, région historique, unité électrique) :")
display(pref.reset_index(drop=True))

Les 39 préfectures du référentiel (région 2022, région historique, unité électrique) :


,nom,region_nom,region_5_nom,population_2022,unite_electrique,taux_herite_du_groupe
0,Tône,Savanes,Savanes,388775,TONE,False
1,Cinkassé,Savanes,Savanes,128959,CINKASSE,False
2,Kpendjal-Ouest,Savanes,Savanes,123330,KPENDJAL + KPENDJAL OUEST,True
3,Kpendjal,Savanes,Savanes,88365,KPENDJAL + KPENDJAL OUEST,True
4,Oti-Sud,Savanes,Savanes,150376,OTI + OTI SUD,True
5,Oti,Savanes,Savanes,124848,OTI + OTI SUD,True
6,Tandjoaré,Savanes,Savanes,138867,TANDJOARE,False
7,Kozah,Kara,Kara,283738,KOZAH,False
8,Assoli,Kara,Kara,66394,ASSOLI,False
9,Bassar,Kara,Kara,152065,BASSAR,False


**Unités d'analyse de l'électricité.** Le fichier électricité ne suit pas les 39 préfectures : il compte **33 unités** (27 préfectures isolées + 6 groupes). Les 12 préfectures des groupes **héritent** du taux de leur groupe (`taux_herite_du_groupe`). Les 33 unités couvrent les 39 préfectures **sans trou ni chevauchement**.

In [7]:
ue = ctx["unites_electriques"]
display(ue[["unite_electrique", "groupe", "n_prefectures", "prefectures", "population_2022", "region_5", "valide_pour_tendance", "motif_tendance_invalide"]])
print("Somme des populations des 33 unités :", fmt_int(ue.population_2022.sum()), "(= total national)")

,unite_electrique,groupe,n_prefectures,prefectures,population_2022,region_5,valide_pour_tendance,motif_tendance_invalide
0,AGOENYIVE,False,1,AGOE-NYIVE,882695,MARITIME,False,taux brut > 100 % (2019: 106.52) ; saut de +40.6 pp en 2019
1,AGOU,False,1,AGOU,85793,PLATEAUX,True,
2,AKEBOU,False,1,AKEBOU,73830,PLATEAUX,True,
3,AMOU,False,1,AMOU,114172,PLATEAUX,True,
4,ASSOLI,False,1,ASSOLI,66394,KARA,True,
5,AVE,False,1,AVE,111214,MARITIME,True,
6,BASSAR,False,1,BASSAR,152065,KARA,True,
7,BINAH,False,1,BINAH,84199,KARA,True,
8,BLITTA,False,1,BLITTA,163272,CENTRALE,False,valeur manquante (2018)
9,CINKASSE,False,1,CINKASSE,128959,SAVANES,True,


Somme des populations des 33 unités : 8 095 498 (= total national)


**Lecture.** 4 unités sont **exclues des tendances 2018→2021** (`valide_pour_tendance` = faux) : Agoè-Nyivé (taux 106,52 % en 2019 et saut de +40,6 pp), Zio (taux 106,84 % en 2021, sauts de +15,3 et +18,8 pp), Yoto (saut de +21,1 pp en 2021) et Blitta (taux 2018 absent). Règle : taux brut > 100 %, saut annuel ≥ 15 pp (seuil paramétré `SEUIL_SAUT_PP`) ou valeur manquante. Les valeurs brutes sont conservées ; `taux_plafonne` (≤ 100) n'est utilisé que pour l'affichage.

In [8]:
ag = ctx["agences"]
print("Agences par service et statut (statut normalisé) :")
display(pd.crosstab([ag.service, ag.categorie], ag.statut, margins=True))
print("Agences OUVERTES accueillant le public (publique ET Utilisé) :")
display(ag.groupby("service")[["ouverte_public", "ouverte_ou_inconnue_public"]].sum().rename(columns={"ouverte_public": "ouvertes (déf. retenue)", "ouverte_ou_inconnue_public": "ouvertes + Nsp (variante)"}))

Agences par service et statut (statut normalisé) :


statut                           En construction  Inachevé  Inconnu (Nsp)  Non applicable (N/a)  Utilisé  All
service categorie                                                                                            
CEET    Agence CEET                            0         0              3                     4       25   32
        Agence CEET - Admin                    0         0              0                     0        1    1
        Agence CEET - Technique                0         0              0                     0        1    1
TdE     Agence TdE                             1         2              7                     1       46   57
        Agence TdE - Admin                     1         0              0                     0        1    2
All                                            2         2             10                     5       74   93

Agences OUVERTES accueillant le public (publique ET Utilisé) :


,ouvertes (déf. retenue),ouvertes + Nsp (variante)
service,,
CEET,25,28
TdE,46,53


In [9]:
ec = ctx["etat_civil_long"]
a = ec[(ec.serie == "A") & (ec.sexe == "Total")]
print("État civil — anomalie détectée par série × année (Total, série A) ; vide = RAS :")
display(a.pivot(index="indicateur", columns="annee", values="anomalie").fillna(""))
print("Exploitable pour une tendance (True/False) :")
display(a.pivot(index="indicateur", columns="annee", values="valide_pour_tendance"))

État civil — anomalie détectée par série × année (Total, série A) ; vide = RAS :


annee,2014,2015,2016,2017,2018,2019
indicateur,,,,,,
Actes de naissance (45 j) non retirés,,,,,,
Enfants enregistrés dans les 45 jours,,,,,,
Enfants scolarisés sans acte de naissance,,aberrante,,,,
Jugements supplétifs,,,,doublon_inter_series,rupture,rupture
Jugements supplétifs transcrits,,,,doublon_inter_series,,
Mort-nés enregistrés en formation sanitaire,,aberrante,,,,
Mort-nés enregistrés hors formation sanitaire,,aberrante,,,,
Naissances enregistrées (total),,,zero_suspect,,zero_suspect,zero_suspect
Naissances enregistrées en formation sanitaire,,,,,,


Exploitable pour une tendance (True/False) :


annee,2014,2015,2016,2017,2018,2019
indicateur,,,,,,
Actes de naissance (45 j) non retirés,True,True,True,True,True,True
Enfants enregistrés dans les 45 jours,True,True,True,True,True,True
Enfants scolarisés sans acte de naissance,True,False,True,True,True,True
Jugements supplétifs,True,True,True,False,True,True
Jugements supplétifs transcrits,True,True,True,False,True,True
Mort-nés enregistrés en formation sanitaire,True,False,True,True,True,True
Mort-nés enregistrés hors formation sanitaire,True,False,True,True,True,True
Naissances enregistrées (total),False,True,False,False,False,False
Naissances enregistrées en formation sanitaire,True,True,True,True,True,True


**Lecture.** Règle objective, identique pour toutes les séries : une valeur est **aberrante** si elle vaut plus de 10× ou moins de 0,2× la médiane des *autres* années ; **rupture** entre 3× et 10× ; **zéro suspect** si un total nul coexiste avec des valeurs positives ; **doublon inter-séries** si deux séries sont identiques la même année. Elle retrouve les anomalies attendues de 2015 (morts-nés hors FS, enfants scolarisés sans acte, naissances vivantes en FS), les totaux nuls de 2016/2018/2019, et signale en plus les **jugements supplétifs 2017** (identiques aux transcrits) ainsi que les **morts-nés en FS 2015** (359 contre ≈ 1 800 les autres années, rapport 0,196 : juste sous le seuil de 0,2, donc dépendant de ce seuil). Les ruptures (jugements 2018–2019…) sont conservées avec avertissement.
**Limite.** La série B (« scolarisés sans acte », 0 à 3 219) porte le même libellé que la série A (313 k à 411 k) : son sens n'est pas documenté, elle est donc **séparée et hors analyse**.

In [10]:
idn = ctx["identite_long"]
print("Documents d'identité — production annuelle (2021 manquant pour CNI, cartes pro et TOTAL, laissé vide) :")
display(idn.pivot(index="annee", columns="libelle", values="valeur_brute")[["Cartes nationales d'identité", "Passeports", "Cartes professionnelles", "Total"]])

Documents d'identité — production annuelle (2021 manquant pour CNI, cartes pro et TOTAL, laissé vide) :


libelle,Cartes nationales d'identité,Passeports,Cartes professionnelles,Total
annee,,,,
2014,153789.0,40756.0,4177.0,198722.0
2015,164411.0,44924.0,4696.0,214031.0
2016,177418.0,47388.0,5820.0,230626.0
2017,187473.0,47825.0,4291.0,239589.0
2018,203357.0,51246.0,4303.0,258906.0
2019,207169.0,68079.0,6037.0,281285.0
2020,179304.0,51767.0,5141.0,236212.0
2021,NaN,66938.0,NaN,NaN
2022,140295.0,74685.0,4351.0,219331.0


---
# Chapitre 03 — Contrôles qualité (data quality & validation)

**Question.** Les données nettoyées sont-elles cohérentes, et que reste-t-il d'imparfait ?
**Méthode.** 30 tests `| Test | Résultat | Statut | Commentaire |` classés `PASS` / `WARNING` / `FAIL`. **Aucun `FAIL` n'est ignoré en silence** ; un `WARNING` signifie « anomalie connue, traitée et documentée ».

In [11]:
tests = run_quality_tests(ctx)
print(tests.statut.value_counts().to_dict())
display(tests[["n", "test", "resultat", "statut", "commentaire"]])
assert not (tests.statut == "FAIL").any(), "Un test FAIL doit être traité avant de continuer"

{'PASS': 19, 'WARNING': 11}


,n,test,resultat,statut,commentaire
0,1,Somme des 6 régions = total national (2022),8 095 498 vs 8 095 498,PASS,
1,2,Somme des 6 blocs = total national (2010),6 191 155 vs 6 191 155,PASS,Lomé est un bloc distinct de la Maritime en 2010
2,3,39 préfectures retrouvées (2022),39,PASS,
3,4,117 communes retrouvées (2022),116 lignes → 117 communes,WARNING,"Corrections documentées (BINAH2, AMOU 2→3, EST-MONO 1→3, TOTOAL AVE) ; `DANYI 1+DANYI ..."
4,5,36 préfectures + Lomé retrouvées (2010),36 préfectures + 5 arrondissements de Lomé,PASS,
5,6,"Parent = somme des enfants (2022 : région←préf., préf.←communes, commune←cantons/quart...",écarts : 0 / 0 / 0,PASS,Vérifié sur 6 + 39 + 116 nœuds
6,7,Parent = somme des enfants (2010),lignes non reconstituées : 0 ; écarts région←préf. : 0,PASS,
7,8,Somme de toutes les feuilles = total national,8 095 498 vs 8 095 498,PASS,Pas de double comptage (6 communes à canton unique)
8,9,Clés hiérarchiques uniques (2022 ; 2010),True ; True,PASS,Format région|préfecture|commune|canton
9,10,"Homonymes (même libellé, plusieurs lignes)",2022 : 13 libellés ; 2010 : 7 libellés,WARNING,Neutralisés par la clé de chemin ; jointure sur nom seul interdite


**Lecture.** Tous les contrôles structurels passent (sommes, clés uniques, couverture 33 ↔ 39, jointures agences à 100 % sur préfecture et commune, coordonnées valides, H + F = Total, TOTAL identité = somme). Les 11 `WARNING` sont des **limites de la donnée**, pas du code : homonymes, commune fusionnée Danyi, taux > 100 %, unités exclues des tendances, **taux régionaux non reproductibles** (Maritime +11,9 pp), date 1076, anomalies d'état civil, lacunes 2021.
**So what.** Le taux d'accès n'ayant pas de dénominateur documenté, « population sans accès » restera un **PROXY**, et le taux national affiché sera le taux **publié** (57,82 %), pas un recalcul.

In [12]:
log = ctx["anomaly_log"]
print(f"anomaly_log : {len(log)} entrées")
display(log.groupby("fichier").size().rename("entrées").to_frame())
display(log[["id", "fichier", "ligne", "champ", "valeur", "nature", "traitement"]])

anomaly_log : 79 entrées


,entrées
fichier,
file-agences-ceet-23-12-2024-11-06-42.csv,7
file-agences-ceet-23-12-2024-11-06-42.csv ; file-agences-togolaise-des-eaux-tde-19-12-2024-17-48-23.csv,3
file-agences-togolaise-des-eaux-tde-19-12-2024-17-48-23.csv,7
observationdata-fsemhj.csv,3
observationdata-kwwolwb.csv,16
observationdata-puncgpd.csv,14
observationdata-sapxctg.csv,9
observationdata-sapxctg.csv ; observationdata-kwwolwb.csv,1
observationdata-vrsicxf.csv,19


,id,fichier,ligne,champ,valeur,nature,traitement
0,1,observationdata-kwwolwb.csv,198,découpage-administratif,BINAH2,Espace manquant,Renommée « BINAH 2 »
1,2,observationdata-kwwolwb.csv,407,découpage-administratif,AMOU 2,Libellé dupliqué dans la préfecture (2e occurrence = commune n°3),Renommée « AMOU 3 » (confirmé par la somme des préfectures)
2,3,observationdata-kwwolwb.csv,432,découpage-administratif,EST-MONO 1,Libellé dupliqué dans la préfecture (2e occurrence = commune n°3),Renommée « EST-MONO 3 » (confirmé par la somme des préfectures)
3,4,observationdata-kwwolwb.csv,512,découpage-administratif,TOTOAL AVE,Faute de frappe (préfixe « TOTOAL »),Renommée « AVE » (préfecture d'Avé)
4,5,observationdata-kwwolwb.csv,366,découpage-administratif,DANYI 1+DANYI 2,Une seule ligne pour 2 communes (DANYI 1 et DANYI 2),"Conservée telle quelle, drapeau `commune_fusionnee` ; non séparable sans inventer"
5,6,observationdata-kwwolwb.csv,547,découpage-administratif,KELEGOUGAN1,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
6,7,observationdata-kwwolwb.csv,548,découpage-administratif,ATIEGOU1,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
7,8,observationdata-kwwolwb.csv,551,découpage-administratif,KELEGOUGAN 2,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
8,9,observationdata-kwwolwb.csv,558,découpage-administratif,ATIEGOU 2,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)
9,10,observationdata-kwwolwb.csv,567,découpage-administratif,KELEGOUGAN 3,Quartier (et non commune) finissant par un chiffre,Classé comme feuille (quartier du DAGL)


In [13]:
files = export_outputs(ctx, tests, TMP)
taille = pd.Series({k: v / 1024 for k, v in files.items()}, name="Ko").round(1)
display(taille.to_frame())
print(f"Volume total écrit : {taille.sum():.1f} Ko | temps clean_data : {TIMINGS['clean_data']:.2f} s")
# relecture : le Parquet est identique au CSV
p = pd.read_parquet(TMP / "dim_territoire.parquet"); c = pd.read_csv(TMP / "dim_territoire.csv")
print("dim_territoire : parquet == csv (lignes, colonnes) :", p.shape == c.shape, p.shape)

,Ko
dim_territoire.csv,125.2
dim_territoire.parquet,56.8
canton_2022.csv,55.0
population_2010.csv,59.3
electricite_long.csv,18.3
unites_electriques.csv,2.4
agences.csv,29.1
etat_civil_long.csv,24.1
identite_long.csv,2.0
anomaly_log.csv,18.2


Volume total écrit : 394.6 Ko | temps clean_data : 0.46 s


dim_territoire : parquet == csv (lignes, colonnes) : True (759, 20)


---
# Chapitre 04 — Démographie 2022 : taille, concentration et cantons

**Question.** Comment la population se répartit-elle entre régions, préfectures, communes et **cantons**, et à quel point quelques territoires concentrent-ils les habitants ?
**Données.** `dim_territoire` (RGPH 2022, 8 095 498 habitants). **Méthode.** Poids relatifs, rangs, statistiques de taille (min, quartiles, médiane, max), part du Top 5/10/20, **courbe de Lorenz et indice de Gini** (0 = tailles identiques, 1 = un seul territoire concentre tout). Les **quartiers du Grand Lomé** sont traités à part des cantons.
**Étiquettes :** tailles et rangs = `OBSERVATION` ; poids, Gini, concentration = `CALCUL`.

In [14]:
dim = ctx["dim_territoire"]
t0 = time.perf_counter(); D = compute_demography(dim); t_demo = time.perf_counter() - t0
print(f"compute_demography : {t_demo*1000:.0f} ms")
k = D["kpi"]
display(pd.Series({"Population totale": fi(k["population_totale"]), "Part du Grand Lomé": fp(k["part_grand_lome_pct"]), "Part du Top 10 des communes": fp(k["part_top10_communes_pct"]),
                   "Commune la plus peuplée": f"{k['commune_plus_peuplee']} ({fi(k['pop_commune_plus_peuplee'])})", "Commune la moins peuplée": f"{k['commune_moins_peuplee']} ({fi(k['pop_commune_moins_peuplee'])})",
                   "Rapport max / min (communes)": f"{fd(k['rapport_max_min_communes'],1)} ×", "Médiane communale": fi(k["mediane_commune"]), "Gini communes / préfectures": f"{fd(k['gini_communes'],2)} / {fd(k['gini_prefectures'],2)}"}, name="KPI").to_frame())
print("Distribution des tailles par niveau :")
display(D["distribution"].round(2))
print("Concentration :")
display(D["concentration"].round(3))

compute_demography : 113 ms


,KPI
Population totale,8 095 498
Part du Grand Lomé,"27,0 %"
Part du Top 10 des communes,"28,7 %"
Commune la plus peuplée,Golfe 1 (351 550)
Commune la moins peuplée,Assoli 2 (9 933)
Rapport max / min (communes),"35,4 ×"
Médiane communale,52 302
Gini communes / préfectures,"0,40 / 0,43"


Distribution des tailles par niveau :


,niveau,n,minimum,q1,mediane,q3,maximum,moyenne,rapport_max_min,gini
0,Préfectures (39),39.0,40240.0,92682.5,138867.0,212498.0,1305681.0,207576.87,32.45,0.43
1,Communes (116 lignes),116.0,9933.0,35790.5,52302.5,76383.5,351550.0,69788.78,35.39,0.40
2,Cantons hors Grand Lomé,375.0,1092.0,6220.0,10646.0,18592.5,161176.0,15752.33,147.60,0.46
3,Quartiers du Grand Lomé,222.0,132.0,4172.0,7292.5,12506.0,62588.0,9857.55,474.15,0.44


Concentration :


,niveau,indicateur,valeur,unite
0,Préfectures (39),Top 5,41.780,% de la population nationale
1,Préfectures (39),Top 10,57.137,% de la population nationale
2,Préfectures (39),Top 20,77.596,% de la population nationale
3,Préfectures (39),Gini,0.428,indice 0–1
4,Communes (116 lignes),Top 5,17.857,% de la population nationale
5,Communes (116 lignes),Top 10,28.712,% de la population nationale
6,Communes (116 lignes),Top 20,43.818,% de la population nationale
7,Communes (116 lignes),Gini,0.397,indice 0–1
8,Cantons hors Grand Lomé,Top 5,9.591,% de la population nationale
9,Cantons hors Grand Lomé,Top 10,15.427,% de la population nationale


In [15]:
print("6 régions 2022 et 5 régions historiques (Maritime incluant le Grand Lomé) :")
display(D["region"][["decoupage", "nom", "population_2022", "poids_national_pct", "nb_prefectures", "rang"]].round(2))

6 régions 2022 et 5 régions historiques (Maritime incluant le Grand Lomé) :


,decoupage,nom,population_2022,poids_national_pct,nb_prefectures,rang
0,6 régions 2022,Savanes,1143520,14.13,7,4
1,6 régions 2022,Kara,985512,12.17,7,5
2,6 régions 2022,Centrale,795529,9.83,5,6
3,6 régions 2022,Plateaux,1635946,20.21,12,2
4,6 régions 2022,Maritime,1346615,16.63,6,3
5,6 régions 2022,Grand Lomé (DAGL),2188376,27.03,2,1
6,5 régions historiques,Centrale,795529,9.83,5,5
7,5 régions historiques,Kara,985512,12.17,7,4
8,5 régions historiques,Maritime (avec Grand Lomé),3534991,43.67,8,1
9,5 régions historiques,Plateaux,1635946,20.21,12,2


In [16]:
pref = D["prefecture"]; com = D["commune"]; can = D["canton"]
plot_classement(pref.assign(id_aff=pref.territoire_id), "préfectures", 15, id_col="id_aff").show()
plot_lorenz(com.population_2022, "communes").show()
plot_boxplot_tailles({"Communes": com, "Cantons": can[can.niveau_feuille == "canton"], "Quartiers du Grand Lomé": can[can.niveau_feuille == "quartier"]}).show()

In [17]:
g = D["concentration"].set_index(["niveau", "indicateur"]).valeur
display(Markdown(f'''
**Lecture.** Les **15 préfectures les plus peuplées** regroupent l'essentiel de la population, et le **Grand Lomé** à lui seul pèse {fp(k['part_grand_lome_pct'])}. Entre communes, le rapport entre la plus grande ({k['commune_plus_peuplee']}) et la plus petite ({k['commune_moins_peuplee']}) est de **{fd(k['rapport_max_min_communes'],1)} ×** ; le Gini des communes vaut **{fd(k['gini_communes'],2)}** et celui des préfectures {fd(k['gini_prefectures'],2)}. Les 10 plus grandes communes regroupent {fp(g[('Communes (116 lignes)','Top 10')])} de la population.

**So what.** Une règle d'allocation **identique par commune** ne suit pas les besoins : elle sous-dote les très grandes communes et sur-dote les très petites. Un critère par habitant, complété par un plancher pour les très petits territoires, est plus cohérent avec cette répartition.
'''))


**Lecture.** Les **15 préfectures les plus peuplées** regroupent l'essentiel de la population, et le **Grand Lomé** à lui seul pèse 27,0 %. Entre communes, le rapport entre la plus grande (Golfe 1) et la plus petite (Assoli 2) est de **35,4 ×** ; le Gini des communes vaut **0,40** et celui des préfectures 0,43. Les 10 plus grandes communes regroupent 28,7 % de la population.

**So what.** Une règle d'allocation **identique par commune** ne suit pas les besoins : elle sous-dote les très grandes communes et sur-dote les très petites. Un critère par habitant, complété par un plancher pour les très petits territoires, est plus cohérent avec cette répartition.


### 04.b Les cantons : tailles et petits cantons

**Question.** Combien de cantons y a-t-il par commune, et combien sont très petits ? **Méthode.** Un canton est « petit » s'il compte moins de `SEUIL_PETIT_CANTON` habitants. Le seuil de **5 000** est un choix documenté : il se situe entre le 10ᵉ centile et le 1ᵉʳ quartile des cantons ; on affiche la **sensibilité** à ce choix. Les quartiers du Grand Lomé sont exclus de ce décompte.

In [18]:
c = can[can.niveau_feuille == "canton"].population_2022
print("Distribution des cantons hors Grand Lomé (habitants) :", c.describe(percentiles=[.1, .25, .5, .75, .9]).round(0).to_dict())
sens = pd.DataFrame([{"seuil (hab.)": s, "cantons concernés": int((c < s).sum()), "part des cantons (%)": round((c < s).mean() * 100, 1),
                      "part de la population des cantons (%)": round(c[c < s].sum() / c.sum() * 100, 1)} for s in (2000, 3000, SEUIL_PETIT_CANTON, 10000)])
display(sens)
print("Cantons par commune (hors Grand Lomé) et petits cantons par préfecture :")
display(com[com.niveau_feuille == "canton"].nb_feuilles.describe().round(1).to_frame("nb de cantons par commune").T)
display(pref.sort_values("nb_petits_cantons", ascending=False)[["nom", "region_nom", "population_2022", "nb_feuilles", "nb_petits_cantons"]].head(8))

Distribution des cantons hors Grand Lomé (habitants) : {'count': 375.0, 'mean': 15752.0, 'std': 17412.0, 'min': 1092.0, '10%': 3556.0, '25%': 6220.0, '50%': 10646.0, '75%': 18592.0, '90%': 30848.0, 'max': 161176.0}


,seuil (hab.),cantons concernés,part des cantons (%),part de la population des cantons (%)
0,2000,10,2.7,0.3
1,3000,25,6.7,0.9
2,5000,68,18.1,3.8
3,10000,176,46.9,17.5


Cantons par commune (hors Grand Lomé) et petits cantons par préfecture :


,count,mean,std,min,25%,50%,75%,max
nb de cantons par commune,103.0,3.6,1.8,1.0,2.5,3.0,4.0,10.0


,nom,region_nom,population_2022,nb_feuilles,nb_petits_cantons
16,Blitta,Centrale,163272,21,9
12,Doufelgou,Kara,84767,15,8
28,Amou,Plateaux,114172,14,6
23,Kloto,Plateaux,145986,12,6
22,Agou,Plateaux,85793,12,5
14,Tchaoudjo,Centrale,240360,13,5
17,Sotouboua,Centrale,138864,12,4
24,Kpélé,Plateaux,80939,9,3


In [19]:
display(Markdown(f'''
**Lecture.** Avec le seuil de {fi(k['seuil_petit_canton'])} habitants, **{k['n_petits_cantons']} cantons** sont petits, soit {fp(k['part_petits_cantons_pct'])} des cantons, mais seulement **{fp(k['part_pop_petits_cantons_pct'])} de la population des cantons**. Le résultat dépend du seuil (tableau ci-dessus) : à 3 000 habitants ils ne sont plus que 25, à 10 000 près de la moitié.

**So what.** Les petits cantons sont nombreux mais peu peuplés : leur enjeu est l'**accès de proximité** (équipements à coût fixe) plus que le volume de population.
**Limites.** 6 communes n'ont qu'un seul canton (commune = canton). Les cantons ne sont pas comparables entre 2010 et 2022 par leur nom (libellés hétérogènes) : voir chapitre 05.
'''))


**Lecture.** Avec le seuil de 5 000 habitants, **68 cantons** sont petits, soit 18,1 % des cantons, mais seulement **3,8 % de la population des cantons**. Le résultat dépend du seuil (tableau ci-dessus) : à 3 000 habitants ils ne sont plus que 25, à 10 000 près de la moitié.

**So what.** Les petits cantons sont nombreux mais peu peuplés : leur enjeu est l'**accès de proximité** (équipements à coût fixe) plus que le volume de population.
**Limites.** 6 communes n'ont qu'un seul canton (commune = canton). Les cantons ne sont pas comparables entre 2010 et 2022 par leur nom (libellés hétérogènes) : voir chapitre 05.


---
# Chapitre 05 — Évolution 2010–2022 à découpage maîtrisé

**Question.** Quels territoires gagnent le plus d'habitants, **sans se laisser piéger par les changements de découpage** ?
**Données.** `dim_territoire` (2022) et `population_2010` (2010). **Méthode.** On ne compare jamais deux territoires sur leur seul nom : on construit un **crosswalk** et des **unités de comparaison** dont le périmètre est identique aux deux dates, ou reconstitué par cantons. Chaque unité reçoit un flag : **HIGH** (comparable), **MEDIUM** (à lire avec prudence), **LOW** (non comparable, jamais classé). TCAM = taux de croissance annuel moyen sur 12 ans.

In [20]:
t0 = time.perf_counter(); E = build_crosswalk_2010_2022(dim, ctx["dim_2010"]); t_cw = time.perf_counter() - t0
print(f"build_crosswalk_2010_2022 : {t_cw*1000:.0f} ms")
cw, cp, tr = E["crosswalk"], E["comparaison"], E["transferts"]
l10 = E["cantons_2010"]
print(f"Cantons 2010 (hors Lomé) retrouvés à l'identique en 2022 : {int(l10.apparie.sum())}/{len(l10)} ({l10.apparie.mean():.0%}) ; part de la population 2010 couverte : {l10.loc[l10.apparie,'population_2010'].sum()/l10.population_2010.sum():.0%}")
print("Transferts de cantons entre préfectures détectés par heuristique de nom, puis examinés :")
display(tr[["canton_2010", "pref_2010", "pref_2022", "verdict", "justification"]])

build_crosswalk_2010_2022 : 240 ms
Cantons 2010 (hors Lomé) retrouvés à l'identique en 2022 : 352/438 (80%) ; part de la population 2010 couverte : 84%
Transferts de cantons entre préfectures détectés par heuristique de nom, puis examinés :


,canton_2010,pref_2010,pref_2022,verdict,justification
0,ADETI KOPE,ZIO,AGOE-NYIVE,CONFIRMÉ,Absent de Zio 2022 ; ADETIKOPE-CENTRE figure sous Agoè-Nyivé 2022 (20 238 hab. en 2010...
1,KAMINA AKEBOU,AKEBOU,EST-MONO,REJETÉ (homonyme),Akébou 2022 contient déjà un canton KAMINA ; KAMINA sous Est-Mono est un autre lieu.
2,AGOTIME SUD/ ADZAKPA,AGOU,GOLFE,REJETÉ (homonyme),"AGOTIME sous Golfe/Agoè-Nyivé est un quartier de Lomé homonyme, pas le canton d'Agou."
3,AGOTIME SUD/ ADZAKPA,AGOU,AGOE-NYIVE,REJETÉ (homonyme),"AGOTIME sous Golfe/Agoè-Nyivé est un quartier de Lomé homonyme, pas le canton d'Agou."
4,ASSAHOUN FIAGBE,AGOU,AVE,REJETÉ (orthographe),"Agou 2022 contient ADJAHUN FIAGBE (même lieu, autre graphie) ; ASSAHOUN/ANDO sous Avé ..."


**Lecture.** L'heuristique repère 4 cantons candidats (5 rapprochements) ; **un seul est confirmé** : le canton `ADETI KOPE` (Zio en 2010) figure sous Agoè-Nyivé en 2022. Les 3 autres sont des homonymes ou des variantes d'orthographe. Zio est donc classé `MEDIUM`, et le Grand Lomé reste `HIGH` avec une réserve chiffrée (ci-dessous).

In [21]:
print("Crosswalk : 39 préfectures 2022 + 3 lignes d'union")
display(cw[["id_comparaison", "niveau_comparaison", "territoire_2022", "territoires_2010", "type_correspondance", "comparability_flag", "couverture_cantons_pct", "pop_2010", "pop_2022", "tcam_pct", "dans_classement"]].round(2))
print("Répartition des flags (lignes préfecture) :", cw[cw.niveau_comparaison == "préfecture"].comparability_flag.value_counts().to_dict())

Crosswalk : 39 préfectures 2022 + 3 lignes d'union


,id_comparaison,niveau_comparaison,territoire_2022,territoires_2010,type_correspondance,comparability_flag,couverture_cantons_pct,pop_2010,pop_2022,tcam_pct,dans_classement
0,C01,préfecture,Tône,Tône,exacte,HIGH,94.02,286029.0,388775,2.59,True
1,C02,préfecture,Cinkassé,Cinkassé,exacte,HIGH,100.00,78592.0,128959,4.21,True
2,C03,préfecture,Kpendjal-Ouest,Kpendjal 2010 : cantons de Kpendjal-Ouest,scission,MEDIUM,100.00,93513.0,123330,2.33,True
3,C04,préfecture,Kpendjal,Kpendjal 2010 : cantons de Kpendjal,scission,HIGH,100.00,61578.0,88365,3.06,True
4,C05,préfecture,Oti-Sud,Oti 2010 : cantons de Oti-Sud,scission,HIGH,100.00,97274.0,150376,3.70,True
5,C06,préfecture,Oti,Oti 2010 : cantons de Oti,scission,HIGH,100.00,93269.0,124848,2.46,True
6,C07,préfecture,Tandjoaré,Tandjoaré,exacte,HIGH,95.38,117969.0,138867,1.37,True
7,C08,préfecture,Kozah,Kozah,exacte,HIGH,93.81,225259.0,283738,1.94,True
8,C09,préfecture,Assoli,Assoli,exacte,MEDIUM,88.56,51491.0,66394,2.14,True
9,C10,préfecture,Bassar,Bassar,exacte,MEDIUM,88.66,119717.0,152065,2.01,True


Répartition des flags (lignes préfecture) : {'HIGH': 20, 'MEDIUM': 17, 'LOW': 2}


In [22]:
u = cp[cp.niveau == "unité de comparaison"]
tn = float(tcam(u.pop_2010.sum(), u.pop_2022.sum()))
print(f"Partition comparable : {len(u)} unités | Σ pop 2010 = {fi(u.pop_2010.sum())} (RGPH 2010 : {fi(ctx['dim_2010'].population_2010.iloc[0])}) | Σ pop 2022 = {fi(u.pop_2022.sum())} (RGPH 2022 : {fi(dim.population_2022.iloc[0])})")
assert u.pop_2010.sum() == ctx["dim_2010"].population_2010.iloc[0] and u.pop_2022.sum() == dim.population_2022.iloc[0]
assert abs(u.contribution_croissance_pct.sum() - 100) < 1e-9
print(f"Σ des contributions = {u.contribution_croissance_pct.sum():.4f} % | TCAM national = {tn:.3f} %/an | flags : {u.comparability_flag.value_counts().to_dict()}")
print("Sensibilité au transfert d'Adéti Kopé (ESTIMATION, indicative) :")
display(cw[cw.note_sensibilite != ""][["territoire_2022", "tcam_pct", "tcam_sensibilite_pct", "note_sensibilite"]].round(2))

Partition comparable : 38 unités | Σ pop 2010 = 6 191 155 (RGPH 2010 : 6 191 155) | Σ pop 2022 = 8 095 498 (RGPH 2022 : 8 095 498)
Σ des contributions = 100.0000 % | TCAM national = 2.260 %/an | flags : {'HIGH': 21, 'MEDIUM': 17}
Sensibilité au transfert d'Adéti Kopé (ESTIMATION, indicative) :


,territoire_2022,tcam_pct,tcam_sensibilite_pct,note_sensibilite
31,Zio,4.41,5.02,ESTIMATION : base 2010 de Zio privée d'Adéti Kopé (20 238 hab.) → croissance à périmèt...
39,Grand Lomé (Golfe + Agoè-Nyivé),2.80,2.73,ESTIMATION : population 2022 privée du seul quartier ADETIKOPE-CENTRE (18 033 hab.) → ...


In [23]:
plot_tcam(u[u.classement_autorise], tn, top_n=20).show()
plot_contributions(u[u.classement_autorise], top_n=12).show()
plot_regions_evolution(cp[cp.niveau == "région historique (5)"], "5 régions historiques").show()

In [24]:
r5 = cp[cp.niveau == "région historique (5)"].sort_values("tcam_pct", ascending=False)
gl = u[u.nom.str.startswith("Grand Lomé")].iloc[0]; top = u[u.classement_autorise].sort_values("tcam_pct", ascending=False).iloc[0]
rap = u[u.classement_autorise & (u.tcam_pct > tn)]; plus = len(rap); n_ok = int(u.classement_autorise.sum()); par_reg = rap.region_6.value_counts().to_dict()
display(Markdown(f'''
**Lecture.** Le pays gagne **{fd((u.pop_2022.sum()-u.pop_2010.sum())/1e6,2)} million d'habitants** en 12 ans (**{fd(tn,2)} %/an**). Le **Grand Lomé** explique {fp(gl.contribution_croissance_pct,0)} de cette hausse pour {fp(gl.pop_2022/u.pop_2022.sum()*100,0)} de la population. L'unité qui croît le plus vite est **{top.nom}** ({fd(top.tcam_pct,1)} %/an) ; {plus} unités sur {n_ok} dépassent le rythme national. Parmi les 5 régions historiques, **{r5.iloc[0].nom}** croît le plus vite ({fd(r5.iloc[0].tcam_pct,1)} %/an) et **{r5.iloc[-1].nom}** le moins vite ({fd(r5.iloc[-1].tcam_pct,1)} %/an).

**So what.** La croissance rapide ne se limite pas à la capitale : parmi les {plus} unités plus rapides que le pays, **{par_reg.get('SAVANES',0)} sont dans les Savanes** et {par_reg.get('MARITIME',0)} dans la Maritime hors Grand Lomé ; elles sont à surveiller en priorité quand on croisera ces rythmes avec l'accès aux services (étape suivante).
**Limites.** (1) Seules **deux dates** (2010, 2022) : aucune tendance intermédiaire. (2) Les **communes n'existent pas en 2010** : aucune comparaison communale. (3) **Golfe et Agoè-Nyivé** ne sont comparables qu'en bloc (Grand Lomé), jamais séparément (LOW). (4) Le TCAM de Zio est probablement **sous-estimé** (transfert d'Adéti Kopé), celui du Grand Lomé légèrement sur-estimé : voir la sensibilité.
'''))


**Lecture.** Le pays gagne **1,90 million d'habitants** en 12 ans (**2,26 %/an**). Le **Grand Lomé** explique 32 % de cette hausse pour 27 % de la population. L'unité qui croît le plus vite est **Zio** (4,4 %/an) ; 15 unités sur 38 dépassent le rythme national. Parmi les 5 régions historiques, **Savanes** croît le plus vite (2,7 %/an) et **Plateaux** le moins vite (1,5 %/an).

**So what.** La croissance rapide ne se limite pas à la capitale : parmi les 15 unités plus rapides que le pays, **6 sont dans les Savanes** et 2 dans la Maritime hors Grand Lomé ; elles sont à surveiller en priorité quand on croisera ces rythmes avec l'accès aux services (étape suivante).
**Limites.** (1) Seules **deux dates** (2010, 2022) : aucune tendance intermédiaire. (2) Les **communes n'existent pas en 2010** : aucune comparaison communale. (3) **Golfe et Agoè-Nyivé** ne sont comparables qu'en bloc (Grand Lomé), jamais séparément (LOW). (4) Le TCAM de Zio est probablement **sous-estimé** (transfert d'Adéti Kopé), celui du Grand Lomé légèrement sur-estimé : voir la sensibilité.


In [25]:
t0 = time.perf_counter(); R = export_demo_evolution(dim, ctx["dim_2010"], TMP); t_exp = time.perf_counter() - t0
taille = pd.Series({k_: v / 1024 for k_, v in R["tailles"].items()}, name="Ko").round(1)
display(taille.to_frame())
pq = sum(v for k_, v in R["tailles"].items() if k_.endswith(".parquet")) / 1024
print(f"export_demo_evolution : {t_exp*1000:.0f} ms | Parquet de l'étape 2 : {pq:.1f} Ko | calcul : compute_demography {t_demo*1000:.0f} ms, build_crosswalk_2010_2022 {t_cw*1000:.0f} ms")

,Ko
fact_commune.csv,20.4
fact_commune.parquet,22.9
canton_2022.csv,77.2
canton_2022.parquet,44.6
demographie_prefecture.csv,6.8
demographie_prefecture.parquet,17.5
demographie_region.csv,1.0
demographie_region.parquet,6.6
crosswalk_2010_2022.csv,9.9
crosswalk_2010_2022.parquet,16.5


export_demo_evolution : 387 ms | Parquet de l'étape 2 : 123.3 Ko | calcul : compute_demography 113 ms, build_crosswalk_2010_2022 240 ms


---
# Chapitre 06 — Services de base : électricité et agences

**Question.** Où l'accès à l'électricité est-il le plus faible, combien de personnes sont concernées, et où les agences CEET et TdE manquent-elles ?
**Règles.** (1) On sépare le **taux** du **nombre de personnes** : *population sans accès = population 2022 × (1 − taux 2021 plafonné)*, un **PROXY** (le dénominateur du taux n'est pas documenté). (2) Une agence est **ouverte** si elle accueille le public et a le statut `Utilisé` ; sites administratifs/techniques à part ; statut inconnu ≠ ouverte. (3) Pour un groupe d'unités : agences = somme, population = somme. (4) Les préfectures d'un groupe **héritent** du taux du groupe.

In [26]:
fp_ = construire_fact_prefecture(dim, ctx["electricite_long"], ctx["unites_electriques"], ctx["agences"], E["crosswalk"])
fu_ = construire_fact_unite(fp_, ctx["unites_electriques"])
print(f"fact_prefecture : {fp_.shape} | fact_unite_electrique : {fu_.shape} | Σ population des unités = {FI(fu_.population_2022.sum())} (RGPH 2022 : {FI(dim.population_2022.iloc[0])})")
assert fu_.population_2022.sum() == dim.population_2022.iloc[0] and int(fp_.agences_ouvertes_total.sum()) == int(ctx["agences"].ouverte_public.sum()) == 71
cols = ["unite_electrique", "population_2022", "taux_plaf_2021", "ecart_national_pp_2021", "pop_sans_acces_2021", "delta_pp_2018_2021", "valide_pour_tendance", "agences_ouvertes_total", "agences_par_100k_total"]
print("Les 8 unités les moins desservies en 2021 :")
display(fu_.sort_values("taux_plaf_2021")[cols].head(8).round(2))
print("Les 8 unités avec le plus de personnes sans accès (PROXY) :")
display(fu_.sort_values("pop_sans_acces_2021", ascending=False)[cols].head(8).round(2))

fact_prefecture : (39, 53) | fact_unite_electrique : (33, 53) | Σ population des unités = 8 095 498 (RGPH 2022 : 8 095 498)
Les 8 unités les moins desservies en 2021 :


,unite_electrique,population_2022,taux_plaf_2021,ecart_national_pp_2021,pop_sans_acces_2021,delta_pp_2018_2021,valide_pour_tendance,agences_ouvertes_total,agences_par_100k_total
11,KPENDJAL + KPENDJAL OUEST,211695,1.86,-55.96,207757.47,-0.24,True,1,0.47
20,TANDJOARE,138867,2.29,-55.53,135686.95,-0.16,True,1,0.72
30,AKEBOU,73830,3.74,-54.08,71068.76,1.13,True,0,0.00
16,EST MONO,164460,5.26,-52.56,155809.40,1.20,True,1,0.61
14,DANKPEN,185662,6.26,-51.56,174039.56,3.33,True,1,0.54
29,KPELE,80939,9.31,-48.51,73403.58,-1.60,True,0,0.00
22,KERAN,128687,9.54,-48.28,116410.26,3.32,True,2,1.55
25,WAWA,101300,12.98,-44.84,88151.26,1.86,True,2,1.97


Les 8 unités avec le plus de personnes sans accès (PROXY) :


,unite_electrique,population_2022,taux_plaf_2021,ecart_national_pp_2021,pop_sans_acces_2021,delta_pp_2018_2021,valide_pour_tendance,agences_ouvertes_total,agences_par_100k_total
4,HAHO + MOYEN MONO,395601,18.68,-39.14,321702.73,8.02,True,4,1.01
5,TONE,388775,33.45,-24.37,258729.76,9.66,True,1,0.26
3,OGOU + ANIE,433625,41.35,-16.47,254321.06,13.48,True,2,0.46
8,OTI + OTI SUD,275224,23.73,-34.09,209913.34,7.88,True,3,1.09
11,KPENDJAL + KPENDJAL OUEST,211695,1.86,-55.96,207757.47,-0.24,True,1,0.47
14,DANKPEN,185662,6.26,-51.56,174039.56,3.33,True,1,0.54
6,LACS + BAS-MONO,336107,50.68,-7.14,165767.97,15.15,True,0,0.00
16,EST MONO,164460,5.26,-52.56,155809.40,1.20,True,1,0.61


In [27]:
sans_aucun = fp_[fp_.sans_aucun_service]
print("Préfectures sans agence ouverte (définition retenue : publique et Utilisé) :")
print(f"  sans CEET : {int(fp_.ceet_sans_agence.sum())} | sans TdE : {int(fp_.tde_sans_agence.sum())} | sans aucune des deux : {len(sans_aucun)} → {', '.join(sans_aucun.prefecture_nom)}")
er = pd.read_csv(OUT / "electricite_regions.csv")
print("\nCohérence des taux régionaux publiés (2021) avec la moyenne des unités pondérée par la population :")
display(er[["region_publiee", "2021", "taux_2021_pondere_unites", "ecart_pp"]].round(1))

Préfectures sans agence ouverte (définition retenue : publique et Utilisé) :
  sans CEET : 18 | sans TdE : 9 | sans aucune des deux : 7 → Lacs, Yoto, Kpendjal-Ouest, Bas-Mono, Kpélé, Akébou, Danyi

Cohérence des taux régionaux publiés (2021) avec la moyenne des unités pondérée par la population :


,region_publiee,2021,taux_2021_pondere_unites,ecart_pp
0,REGION CENTRALE,36.0,37.2,1.3
1,REGION DE LA KARA,36.6,36.5,-0.2
2,REGION DES PLATEAUX,27.6,28.1,0.5
3,REGION DES SAVANES,22.4,23.3,0.9
4,REGION MARITIME,73.9,85.7,11.9


In [28]:
tn_pays = TAUX_NATIONAL[2021]
t5 = fu_.sort_values("pop_sans_acces_2021", ascending=False).head(5)
pire, meil = fu_.sort_values("taux_plaf_2021").iloc[0], fu_.sort_values("taux_plaf_2021").iloc[-1]
mx = er.loc[er.ecart_pp.abs().idxmax()]
display(Markdown(f'''
**Lecture.** {phrase_sous_moyenne(fp_)} Le taux va de **{FP(pire.taux_plaf_2021)}** ({pire.unite_electrique.title()}) à **{FP(meil.taux_plaf_2021)}** ({meil.unite_electrique.title()}). Les 5 unités comptant le plus de personnes sans accès regroupent **{FI(t5.pop_sans_acces_2021.sum())}** personnes (PROXY), soit {FP(t5.pop_sans_acces_2021.sum() / fu_.pop_sans_acces_2021.sum() * 100, 0)} du total ({FI(fu_.pop_sans_acces_2021.sum())}). **{len(sans_aucun)}** préfectures n'ont aucune agence CEET ni TdE ouverte.

**So what.** Ciblage à deux entrées : le **taux** repère les unités en retard, la **population sans accès** dimensionne l'effort.
**Limites.** Les taux régionaux publiés ne se retrouvent pas à partir des unités (écart maximal : {FD(mx.ecart_pp, 1)} points pour {mx.region_publiee.title()}) : le dénominateur du taux est inconnu, donc « population sans accès » est un ordre de grandeur. Les unités Agoè-Nyivé, Zio, Yoto (saut ou taux > 100 %) et Blitta (2018 absent) sont exclues des tendances.
'''))


**Lecture.** 32 préfectures sur 39 présentent un taux d'accès à l'électricité inférieur à la moyenne nationale (57,8%). Le taux va de **1,9%** (Kpendjal + Kpendjal Ouest) à **100,0%** (Zio). Les 5 unités comptant le plus de personnes sans accès regroupent **1 252 424** personnes (PROXY), soit 34% du total (3 683 089). **7** préfectures n'ont aucune agence CEET ni TdE ouverte.

**So what.** Ciblage à deux entrées : le **taux** repère les unités en retard, la **population sans accès** dimensionne l'effort.
**Limites.** Les taux régionaux publiés ne se retrouvent pas à partir des unités (écart maximal : 11,9 points pour Region Maritime) : le dénominateur du taux est inconnu, donc « population sans accès » est un ordre de grandeur. Les unités Agoè-Nyivé, Zio, Yoto (saut ou taux > 100 %) et Blitta (2018 absent) sont exclues des tendances.


---
# Chapitre 07 — Indice de Priorité Territoriale (IPT)

**Question.** Comment classer objectivement les unités selon la combinaison de leurs déficits ?
**Méthode.** Six composantes orientées « plus grand = plus défavorable », chacune ramenée à 0–100 (winsorisation p5/p95, puis min–max) ; IPT = Σ poids × composante ÷ Σ poids ; niveaux = quartiles de l'IPT. **Unité : 33 unités électriques** (les préfectures d'un groupe ne sont pas dissociables sans inventer de données). Les cas non interprétables (croissance non comparable, tendance invalide) reçoivent la **valeur neutre** (médiane) et sont signalés. Les **pondérations sont une HYPOTHÈSE** : on en mesure la sensibilité au chapitre 08.

In [29]:
display(pd.DataFrame([dict(code=k, composante=v[0], formule=v[1], sens=v[2], nature=v[3]) for k, v in COMPOSANTES.items()]))
sc = pd.DataFrame([dict(scénario=k, **v, total=sum(v.values())) for k, v in SCENARIOS.items()]); display(sc)
ipt_ = compute_ipt(fu_, SCENARIOS[SCENARIO_DEFAUT])
print("Seuils des niveaux (quartiles) :", {k: round(v, 1) for k, v in ipt_.attrs["seuils_classes"].items()}, "| répartition :", ipt_.classe_priorite.value_counts().to_dict())
print(f"Valeurs neutres : croissance {int(ipt_.neutre_c3.sum())} unités, stagnation de l'accès {int(ipt_.neutre_c4.sum())} unités")
display(ipt_.sort_values("ipt", ascending=False)[["unite_electrique", "ipt", "rang_ipt", "classe_priorite", "taux_plaf_2021", "pop_sans_acces_2021", "agences_par_100k_total", "neutre_c3", "neutre_c4"]].head(10).round(1))

,code,composante,formule,sens,nature
0,c1_deficit_acces,Déficit d'accès,100 − taux d'accès à l'électricité 2021 (plafonné à 100),"plus le taux est bas, plus la priorité est forte",CALCUL
1,c2_pop_sans_acces,Population sans accès,population 2022 × (1 − taux 2021),"plus il y a de personnes sans accès, plus la priorité est forte",PROXY
2,c3_croissance,Croissance démographique,"TCAM 2010–2022 (si comparabilité HIGH ou MEDIUM, sinon valeur neutre)","plus la croissance est rapide, plus la priorité est forte",CALCUL
3,c4_stagnation_acces,Stagnation de l'accès,"− (taux 2021 − taux 2018), en points (si la tendance est valide, sinon valeur neutre)","moins l'accès progresse, plus la priorité est forte",CALCUL
4,c5_deficit_equipement,Déficit d'équipement,− agences CEET + TdE ouvertes pour 100 000 habitants,"moins il y a d'agences par habitant, plus la priorité est forte",CALCUL
5,c6_absence_service,Absence de service,moyenne des parts de population vivant sans agence CEET ouverte et sans agence TdE ouv...,"plus la population est privée d'agence, plus la priorité est forte",CALCUL


,scénario,c1_deficit_acces,c2_pop_sans_acces,c3_croissance,c4_stagnation_acces,c5_deficit_equipement,c6_absence_service,total
0,A · Accès,35,15,10,25,8,7,100
1,B · Population,15,35,25,10,8,7,100
2,C · Équilibre,20,20,15,15,15,15,100
3,D · Équipement,10,15,10,5,30,30,100


Seuils des niveaux (quartiles) : {'Q1': 36.1, 'Médiane': 47.4, 'Q3': 66.3} | répartition : {'Faible': 9, 'Moyenne': 8, 'Très haute': 8, 'Haute': 8}
Valeurs neutres : croissance 2 unités, stagnation de l'accès 4 unités


,unite_electrique,ipt,rang_ipt,classe_priorite,taux_plaf_2021,pop_sans_acces_2021,agences_par_100k_total,neutre_c3,neutre_c4
11,KPENDJAL + KPENDJAL OUEST,85.1,1,Très haute,1.9,207757.5,0.5,False,False
14,DANKPEN,75.8,2,Très haute,6.3,174039.6,0.5,False,False
16,EST MONO,73.8,3,Très haute,5.3,155809.4,0.6,False,False
30,AKEBOU,73.3,4,Très haute,3.7,71068.8,0.0,False,False
5,TONE,72.7,5,Très haute,33.4,258729.8,0.3,False,False
29,KPELE,69.8,6,Très haute,9.3,73403.6,0.0,False,False
20,TANDJOARE,68.1,7,Très haute,2.3,135686.9,0.7,False,False
3,OGOU + ANIE,66.9,8,Très haute,41.4,254321.1,0.5,False,False
8,OTI + OTI SUD,66.3,9,Haute,23.7,209913.3,1.1,False,False
6,LACS + BAS-MONO,64.7,10,Haute,50.7,165768.0,0.0,False,False


---
# Chapitre 08 — Robustesse et corrélations

**Question.** Le classement dépend-il des pondérations choisies ?
**Méthode.** Fréquence en Top 10 sur **4 scénarios + 500 tirages de poids** (Dirichlet α = 1, `random_state` = 42) + variante « statut inconnu compté ouvert » + normalisation par rang. Une unité est **robuste** si elle est dans le Top 10 dans au moins 80 % des configurations. Corrélations de Spearman entre composantes : |ρ| ≥ 0,8 signale un risque de double comptage.

In [30]:
rob_ = robustesse(fu_)
cor_ = correlation_composantes(fu_)
print(f"{rob_.attrs['n_configurations']} configurations | robustes (≥ {SEUIL_ROBUSTE:.0f} %) : {int((rob_.stabilite == 'robuste').sum())} unités")
display(rob_.sort_values("priority_stability_score", ascending=False)[["unite_electrique", "priority_stability_score", "rang_median", "rang_min", "rang_max", "stabilite"]].head(12).round(1))
print("Corrélations de Spearman entre composantes (normalisées) :"); display(cor_["rho"].round(2))
print("Verdict :", cor_["verdict"])

506 configurations | robustes (≥ 80 %) : 5 unités


,unite_electrique,priority_stability_score,rang_median,rang_min,rang_max,stabilite
14,DANKPEN,100.0,4.0,1,9,robuste
11,KPENDJAL + KPENDJAL OUEST,100.0,1.0,1,7,robuste
16,EST MONO,99.4,5.0,2,11,robuste
5,TONE,90.9,6.0,1,18,robuste
30,AKEBOU,88.1,3.0,1,17,robuste
29,KPELE,73.1,5.5,1,25,sensible
20,TANDJOARE,71.7,8.0,2,17,sensible
6,LACS + BAS-MONO,61.7,9.0,1,27,sensible
8,OTI + OTI SUD,56.9,10.0,3,18,sensible
3,OGOU + ANIE,55.9,10.0,2,26,sensible


Corrélations de Spearman entre composantes (normalisées) :


,c1_deficit_acces,c2_pop_sans_acces,c3_croissance,c4_stagnation_acces,c5_deficit_equipement,c6_absence_service
c1_deficit_acces,1.00,0.46,-0.12,0.65,-0.01,0.40
c2_pop_sans_acces,0.46,1.00,0.32,-0.08,0.11,0.32
c3_croissance,-0.12,0.32,1.00,-0.12,0.10,-0.04
c4_stagnation_acces,0.65,-0.08,-0.12,1.00,0.15,0.29
c5_deficit_equipement,-0.01,0.11,0.10,0.15,1.00,0.75
c6_absence_service,0.40,0.32,-0.04,0.29,0.75,1.00


Verdict : Aucune paire de composantes ne dépasse |ρ| = 0,8 : pas de risque de double comptage détecté.


In [31]:
rb = rob_[rob_.stabilite == "robuste"]
display(Markdown(f'''
**Lecture.** **{len(rb)}** unités sur 33 restent dans le Top 10 dans au moins 80 % des {rob_.attrs['n_configurations']} configurations : {liste_fr(rb.unite_electrique.str.title(), 5)}. Les autres sont **sensibles aux pondérations** : leur rang dépend des choix de calcul. {cor_['verdict']}

**So what.** Commencer par les unités robustes ; traiter les autres comme des candidates à confirmer. **Limite.** La robustesse mesure la sensibilité aux **poids**, pas aux **données** : un taux d'accès faux fausserait toutes les configurations.
'''))


**Lecture.** **5** unités sur 33 restent dans le Top 10 dans au moins 80 % des 506 configurations : Tone, Kpendjal + Kpendjal Ouest, Dankpen, Est Mono et Akebou. Les autres sont **sensibles aux pondérations** : leur rang dépend des choix de calcul. Aucune paire de composantes ne dépasse |ρ| = 0,8 : pas de risque de double comptage détecté.

**So what.** Commencer par les unités robustes ; traiter les autres comme des candidates à confirmer. **Limite.** La robustesse mesure la sensibilité aux **poids**, pas aux **données** : un taux d'accès faux fausserait toutes les configurations.


---
# Chapitre 09 — Recommandations et messages clés

**Règle.** DONNÉE → INDICATEUR → COMPARAISON → INTERPRÉTATION → CONCLUSION → RECOMMANDATION. Chaque recommandation indique **où**, **pourquoi** (preuve chiffrée), **quelle action**, **quelle priorité**, **quel horizon** et **quel indicateur de suivi**. Aucun chiffre n'est écrit à la main : tout vient des tables.

In [32]:
reco_ = construire_recommandations(ipt_, fp_, ctx["dim_territoire"], D["kpi"], ctx["etat_civil_long"], ctx["agences"], D["canton"])
print(f"{len(reco_)} recommandations :")
display(reco_[["Axe", "Territoire", "Valeur", "Priorité", "Horizon"]])
print("Exemple, avec sa preuve :"); print(reco_.iloc[0].Recommandation); print("  Preuve :", reco_.iloc[0].Preuve)
msgs = messages_cles(ipt_, fp_, D["commune"], D["kpi"], rob_, cp, ctx["identite_long"])
for m in msgs[:5]: print(f"• {m['constat']}\n    preuve : {m['preuve']}\n    implication : {m['implication']}")

11 recommandations :


,Axe,Territoire,Valeur,Priorité,Horizon
0,Électricité,Haho + Moyen-Mono,"18,7% · 321 703 pers.",Haute,Moyen terme (2–3 ans)
1,Électricité,Tône,"33,5% · 258 730 pers.",Très haute,Moyen terme (2–3 ans)
2,Électricité,Ogou + Anié,"41,4% · 254 321 pers.",Très haute,Moyen terme (2–3 ans)
3,Électricité,Oti-Sud + Oti,"23,7% · 209 913 pers.",Haute,Moyen terme (2–3 ans)
4,Électricité,Kpendjal-Ouest + Kpendjal,"1,9% · 207 757 pers.",Très haute,Moyen terme (2–3 ans)
5,Agences,"Lacs, Yoto, Kpendjal-Ouest, Bas-Mono, Kpélé, Akébou et Danyi",7 préfectures,Haute,Court à moyen terme (1–2 ans)
6,Agences,Golfe,130 568 hab./agence,Haute,Moyen terme (2–3 ans)
7,Agences,Agoè-Nyivé,176 539 hab./agence,Haute,Moyen terme (2–3 ans)
8,Démographie,"Blitta, Doufelgou et Amou (les plus concernées)",68 cantons (18% des cantons),Moyenne,Long terme (3 ans et plus)
9,État civil,National (données non territorialisées),170 266 (2015) → 103 893 (2019) · 411 310 sans acte (2019),Nationale,Court à moyen terme (1–3 ans)


Exemple, avec sa preuve :
Inscrire Haho + Moyen-Mono en tête du programme d'électrification : l'objectif minimal est de rejoindre le taux national (57,8%), soit environ 154 838 personnes supplémentaires à raccorder.
  Preuve : Taux 18,7% (−39,1 pp vs national) ; 395 601 habitants ; rang 1 sur 33 pour la population sans accès.
• La population est très concentrée.
    preuve : Le Grand Lomé pèse 27,0% de la population et les 10 plus grandes communes 28,7%.
    implication : Une dotation identique par commune ne suit pas la population : privilégier des critères par habitant.
• Le Grand Lomé porte une part majeure de la croissance.
    preuve : Il explique 32% de l'accroissement 2010–2022 pour 27% de la population.
    implication : La pression sur les services urbains continue de croître plus vite que dans le reste du pays.
• L'accès à l'électricité est très inégal.
    preuve : 32 préfectures sur 39 sont sous la moyenne nationale (57,8%) ; le taux va de 1,9% à 100,0% selon les unités (ma

---
# Annexe A — Charte du template et contrastes (WCAG)

**Question.** Les couleurs de la charte du *template-modele* sont-elles lisibles ? **Méthode.** Ratio de contraste WCAG 2.1 de chaque paire texte/fond ; cible **≥ 4,5 : 1** pour le texte courant.

In [33]:
tc = table_contrastes()
display(tc)
echecs = tc[~tc.conforme]
print(f"{len(echecs)} paires sous 4,5:1 :", "; ".join(f"{r.paire} ({r.ratio})" for r in echecs.itertuples()))

,paire,texte,fond,ratio,seuil,conforme
0,Texte courant sur fond de page,#1e3a4c,#f7fcfe,11.51,4.5,True
1,Texte courant sur blanc,#1e3a4c,#ffffff,11.90,4.5,True
2,Titres bleus sur blanc,#003f8c,#ffffff,10.05,4.5,True
3,Vert institutionnel sur blanc,#016b4b,#ffffff,6.55,4.5,True
4,"Blanc sur vert institutionnel (navbar, boutons)",#ffffff,#016b4b,6.55,4.5,True
5,Texte vert foncé sur or (menu actif),#004f31,#fdcf00,6.52,4.5,True
6,Blanc sur or,#ffffff,#fdcf00,1.49,4.5,False
7,Or en texte sur blanc,#fdcf00,#ffffff,1.49,4.5,False
8,Rouge du gabarit en texte sur blanc,#f20e36,#ffffff,4.29,4.5,False
9,Rouge foncé ajouté pour le texte,#c4102f,#ffffff,6.07,4.5,True


4 paires sous 4,5:1 : Blanc sur or (1.49); Or en texte sur blanc (1.49); Rouge du gabarit en texte sur blanc (4.29); Gris-bleu du gabarit en texte sur blanc (2.05)


display(Markdown(f'''
**Lecture.** Le texte courant du template est très lisible (≥ 10 : 1 sur le fond de page et sur blanc ; 6,6 : 1 en blanc sur le vert institutionnel). **Quatre couleurs du gabarit ne doivent jamais servir de texte sur fond blanc** : l'or ({tc[tc.paire == 'Or en texte sur blanc'].ratio.iloc[0]} : 1), le rouge ({tc[tc.paire == 'Rouge du gabarit en texte sur blanc'].ratio.iloc[0]} : 1), le gris-bleu ({tc[tc.paire == 'Gris-bleu du gabarit en texte sur blanc'].ratio.iloc[0]} : 1) et le blanc sur or. Le tableau de bord les utilise donc **comme surfaces ou traits** (barres, bordures, contours) et **ajoute deux teintes sombres pour le texte** : rouge foncé `#c4102f` ({tc[tc.paire == 'Rouge foncé ajouté pour le texte'].ratio.iloc[0]} : 1) et gris-bleu foncé `#4f6270` ({tc[tc.paire == 'Gris-bleu foncé ajouté pour les notes'].ratio.iloc[0]} : 1). Sur l'or du menu actif, le texte est vert foncé (6,5 : 1).
**Aucune information n'est portée par la seule couleur** : le niveau de comparabilité figure dans les étiquettes (« · MEDIUM »), le niveau de priorité dans un badge texte, et la sélection ajoute un contour.
'''))

---
## Récapitulatif
- **Fait** : référentiel territorial 2022/2010 validé par somme ; démographie, crosswalk 2010→2022 ; services de base (électricité, agences) ; **IPT sur 33 unités** avec scénarios, corrélations et robustesse ; recommandations chiffrées ; charte du template vérifiée.
- **Limites** : deux recensements seulement ; taux d'accès sans dénominateur documenté (population sans accès = PROXY) ; état civil et identité nationaux ; pondérations de l'IPT = hypothèse ; contours et coordonnées des cartes indicatifs.
- **Application** : `streamlit run app.py` (7 pages) ; validation par `pytest tests` (voir README).